In [ ]:
import scrapy


class BooksSpiderSpider(scrapy.Spider):
    name = "books_spider"
    allowed_domains = ["books.toscrape.com"]
    start_urls = ["https://books.toscrape.com/catalogue/page-1.html"]

    page_count = 1
    max_pages = 5

    def parse(self, response):

        books = response.css("article.product_pod")

        for book in books:
            book_url = book.css("h3 a::attr(href)").get()

            yield response.follow(
                book_url,
                callback=self.parse_book
            )

        if self.page_count < self.max_pages:
            next_page = response.css("li.next a::attr(href)").get()

            if next_page:
                self.page_count += 1
                yield response.follow(next_page, callback=self.parse)

    def parse_book(self, response):

        yield {
            "title": response.css("div.product_main h1::text").get(),
            "category": response.css("ul.breadcrumb li:nth-child(3) a::text").get(),
            "price": response.css("p.price_color::text").get(),
            "rating": response.css("p.star-rating::attr(class)").get().split()[-1],
            "availability": " ".join(
                [i.strip() for i in response.css("p.instock.availability::text").getall() if i.strip()]
            ),
            "description": response.css("#product_description + p::text").get(default=""),
            "upc": response.xpath('//th[text()="UPC"]/following-sibling::td/text()').get(),
            "number_of_reviews": response.xpath('//th[text()="Number of reviews"]/following-sibling::td/text()').get(),
            "product_url": response.url,
        }

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# Load cleaned dataset
df = pd.read_csv("books_cleaned.csv")

# -----------------------------
# Summary Statistics
# -----------------------------

print("="*60)
print("SUMMARY STATISTICS")
print("="*60)

print(df.describe(include='all'))

print("\nCategory Counts\n")
print(df["category"].value_counts())

print("\nHighest Rated Books\n")
print(df[df["rating"]==5][["title","category","price"]])

print("\nStock Statistics\n")
print(df["stock_count"].describe())

print("\nMissing Values\n")
print(df.isnull().sum())

# Graph 1 - Price Distribution
plt.figure(figsize=(8,5))
plt.hist(df["price"], bins=10)

plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Number of Books")

plt.tight_layout()
plt.savefig("price_distribution.png")
plt.show()

# Graph 2 - Rating Distribution

plt.figure(figsize=(8,5))

df["rating"].value_counts().sort_index().plot(kind="bar")

plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Books")

plt.tight_layout()
plt.savefig("rating_distribution.png")
plt.show()

# Graph 3 - Average Price by Category
plt.figure(figsize=(12,6))

df.groupby("category")["price"].mean().sort_values().plot(kind="bar")

plt.title("Average Price by Category")
plt.xlabel("Category")
plt.ylabel("Average Price")

plt.xticks(rotation=90)

plt.tight_layout()
plt.savefig("avg_price.png")
plt.show()

# Graph 4 - Price vs Rating
plt.figure(figsize=(8,5))

plt.scatter(df["rating"],df["price"])

plt.title("Price vs Rating")
plt.xlabel("Rating")
plt.ylabel("Price")

plt.tight_layout()
plt.savefig("price_rating.png")
plt.show()


# Word Cloud

text = " ".join(df["description"].astype(str))

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(text)

plt.figure(figsize=(12,6))

plt.imshow(wordcloud)

plt.axis("off")

plt.title("Book Description Word Cloud")

plt.savefig("word_cloud.png")
plt.show()


#Category Pattern
print("\nMost Common Categories")

print(df["category"].value_counts().head())


# Highly Rated Books
print("\nHighest Rated Books")

print(df[df["rating"]==5][["title","category","price"]])


# Stock Pattern
print("\nHighest Stock Books")

print(df.nlargest(10,"stock_count")[["title","stock_count"]])


# Unusual Values
print("\nMost Expensive Books")
print(df.nlargest(5,"price")[["title","price"]])



In [ ]:
import pandas as pd

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("books.csv")

# -----------------------------
# Task 1 Report
# -----------------------------
print("=" * 50)
print("TASK 1 REPORT")
print("=" * 50)

print("\nTotal Records Scraped:")
print(len(df))

print("\nMissing Values:")
print(df.isnull().sum())

duplicate_upc = df["upc"].duplicated().sum()

print("\nDuplicate UPC Values:")
print(duplicate_upc)

# -----------------------------
# Task 2 - Data Cleaning
# -----------------------------

# Remove extra spaces
for col in ["title", "category", "description"]:
    df[col] = df[col].fillna("").str.strip()

# Handle missing descriptions
df["description"] = df["description"].replace("", "No Description Available")

# Remove duplicate books using UPC
df.drop_duplicates(subset="upc", inplace=True)

# Convert price to float
df["price"] = (
    df["price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

# Convert rating into integers
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["rating"].map(rating_map)

# Convert reviews into integer
df["number_of_reviews"] = df["number_of_reviews"].astype(int)

# Extract available stock count
df["stock_count"] = (
    df["availability"]
    .str.extract(r"(\d+)")
    .astype(int)
)

# -----------------------------
# Feature Engineering
# -----------------------------

# Feature 1
df["description_word_count"] = (
    df["description"]
    .apply(lambda x: len(x.split()))
)

# Feature 2
df["is_premium"] = (
    df["price"] > df["price"].median()
)

# Feature 3
df["popularity_score"] = (
    df["rating"] * df["stock_count"]
)

# -----------------------------
# Save Cleaned Dataset
# -----------------------------

df.to_csv("books_cleaned.csv", index=False)

print("\nCleaned dataset saved as books_cleaned.csv")

print("\nFirst Five Rows:\n")
print(df.head())